# `SmoothedLine`

`SmoothedLine` stores a polyline, smooths its coordinates with a Savitzky-Golay filter, builds a parametric spline, and resamples that spline into the final `result`. It also exposes spline-based position and tangent queries.


In [ ]:
import numpy as np
import nematics3d as n3d

t = np.linspace(0.0, 4.0 * np.pi, 121)
coords = np.column_stack([
    t,
    np.sin(t) + 0.08 * np.sin(9.0 * t),
    0.25 * np.cos(0.5 * t),
])
line = n3d.SmoothedLine(
    coords,
    window_length=9,
    order=3,
    min_line_length=2,
)
line.result.shape, line.calc_status


## The smoothing pipeline

The important stages are `raw_coords` -> `calc_coords` -> Savitzky-Golay filtering -> FITPACK spline -> resampled `result`. `entity_tck` stores the spline representation when smoothing succeeds.

`result` is exposed as a read-only NumPy array. The original raw coordinates remain separate from the final output.


In [ ]:
line.calc_is_smoothed, line.entity_tck is not None, line.result.flags.writeable


## `window_length` and `window_ratio`

You can specify either an explicit odd smoothing window with `window_length` or an approximate relative scale with `window_ratio`. After a commit, the two values are synchronized to the actual odd window used by the filter. If an even `window_length` is supplied, it is increased to the next odd integer.


In [ ]:
line.act_commit(window_ratio=15)
line.opts.window_length, line.opts.window_ratio


## Output density

`num_out_ratio` controls only the number of samples taken from the fitted spline. Changing only this option reuses the cached spline rather than rerunning filtering and spline fitting.


In [ ]:
line.act_commit(num_out_ratio=2)
line.calc_num_init, line.calc_num_out, line.result.shape


## Position and tangent queries

Spline locations use `u_percent` in the range `[0, 100]`. `act_calc_pos()` returns the spline position and `act_calc_tangent()` returns a unit tangent.


In [ ]:
position = line.act_calc_pos(50)
tangent = line.act_calc_tangent(50)
position, tangent, np.linalg.norm(tangent)


## `interp` versus `wrap`

`mode="interp"` treats the line as open. `mode="wrap"` treats it as periodic; in that mode `u_percent=100` maps back to the same spline location as `u_percent=0`.


In [ ]:
theta = np.linspace(0.0, 2.0 * np.pi, 120, endpoint=False)
ring = np.column_stack([np.cos(theta), np.sin(theta), 0.2 * np.cos(2.0 * theta)])
periodic_line = n3d.SmoothedLine(
    ring,
    window_length=9,
    order=3,
    min_line_length=2,
    mode="wrap",
)
np.allclose(periodic_line.act_calc_pos(0), periodic_line.act_calc_pos(100))


## Recoverable fallback

Recognized configuration problems do not leave the object half-initialized. Instead, smoothing falls back to the processed raw coordinates, sets `calc_is_smoothed=False`, clears `entity_tck`, and records the reason in `calc_status`. A later valid commit can recover the object and rebuild the spline.


In [ ]:
fallback = n3d.SmoothedLine(
    coords[:20],
    window_length=5,
    order=3,
    min_line_length=50,
)
fallback.calc_is_smoothed, fallback.calc_status


In [ ]:
fallback.act_commit(min_line_length=2)
fallback.calc_is_smoothed, fallback.calc_status


## Updating the input line

Raw-coordinate changes go through the host commit pipeline. Committing new `coords` reapplies the smoothing options and rebuilds all dependent state.


In [ ]:
new_t = np.linspace(0.0, 3.0 * np.pi, 81)
new_coords = np.column_stack([new_t, np.sin(new_t), 0.2 * np.cos(new_t)])
line.act_commit(coords=new_coords)
line.calc_num_init, line.result.shape, line.calc_status


## Scope

This tutorial covers only `SmoothedLine`. Functions sampled along a line are handled separately by `SmoothedLineFunc`.
